In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt

In [ ]:
df = pd.read_parquet("model1_dataset.parquet")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (253, 12)

Columns:
['invoice_id', 'buyer_id', 'issue_date', 'buyer_dbt_mean', 'buyer_dbt_sd', 'buyer_dispute_rate', 'buyer_promise_keep', 'amount_rel', 'terms', 'quarter_end', 'late_target', 'dbt_target']

First 5 rows:


,invoice_id,buyer_id,issue_date,buyer_dbt_mean,buyer_dbt_sd,buyer_dispute_rate,buyer_promise_keep,amount_rel,terms,quarter_end,late_target,dbt_target
0,INV-2113,BUY-06,2026-01-01,0.000000,0.000000,0.0,0.5,1.0,60,0,1,8
1,INV-2139,BUY-07,2026-01-01,8.000000,0.000000,0.0,0.5,1.0,30,0,1,24
2,INV-2237,BUY-18,2026-01-01,16.000000,11.313708,0.0,0.5,1.0,30,0,0,-4
3,INV-2064,BUY-02,2026-01-02,9.333333,14.047538,0.0,0.5,1.0,60,0,1,11
4,INV-2075,BUY-03,2026-01-02,9.750000,11.500000,0.0,0.5,1.0,30,0,1,18


In [ ]:
df["issue_date"] = pd.to_datetime(df["issue_date"])

print("Earliest invoice:", df["issue_date"].min())
print("Latest invoice:", df["issue_date"].max())

print("\nInvoices by date:")
display(df[["invoice_id", "issue_date", "late_target"]].sort_values("issue_date").tail(10))

Earliest invoice: 2026-01-01 00:00:00
Latest invoice: 2026-04-30 00:00:00

Invoices by date:


,invoice_id,issue_date,late_target
243,INV-2114,2026-04-28,0
244,INV-2120,2026-04-28,1
245,INV-2181,2026-04-28,1
248,INV-2145,2026-04-29,1
246,INV-2035,2026-04-29,1
247,INV-2137,2026-04-29,1
249,INV-2191,2026-04-29,1
251,INV-2166,2026-04-30,1
250,INV-2132,2026-04-30,0
252,INV-2239,2026-04-30,1


In [ ]:
# Sort chronologically
df = df.sort_values("issue_date").reset_index(drop=True)

# Last invoice date
latest_date = df["issue_date"].max()

# Six-week test period
test_start = latest_date - pd.Timedelta(weeks=6)

print("Latest invoice date:", latest_date.date())
print("Test period starts:", test_start.date())

# Split by time
train_df = df[df["issue_date"] < test_start].copy()
test_df = df[df["issue_date"] >= test_start].copy()

print("\nTRAINING SET")
print("Rows:", len(train_df))
print("Date range:", train_df["issue_date"].min().date(),
      "to", train_df["issue_date"].max().date())

print("\nTEST SET")
print("Rows:", len(test_df))
print("Date range:", test_df["issue_date"].min().date(),
      "to", test_df["issue_date"].max().date())

print("\nTarget distribution")
print("Train:")
print(train_df["late_target"].value_counts())

print("\nTest:")
print(test_df["late_target"].value_counts())

Latest invoice date: 2026-04-30
Test period starts: 2026-03-19

TRAINING SET
Rows: 167
Date range: 2026-01-01 to 2026-03-18

TEST SET
Rows: 86
Date range: 2026-03-20 to 2026-04-30

Target distribution
Train:
late_target
1    129
0     38
Name: count, dtype: int64

Test:
late_target
1    68
0    18
Name: count, dtype: int64


In [ ]:
FEATURES = [
    "buyer_dbt_mean",
    "buyer_dbt_sd",
    "buyer_dispute_rate",
    "buyer_promise_keep",
    "amount_rel",
    "terms",
    "quarter_end"
]

TARGET = "late_target"

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nFeatures:")
print(FEATURES)

X_train: (167, 7)
y_train: (167,)
X_test : (86, 7)
y_test : (86,)

Features:
['buyer_dbt_mean', 'buyer_dbt_sd', 'buyer_dispute_rate', 'buyer_promise_keep', 'amount_rel', 'terms', 'quarter_end']


In [ ]:
# Build the baseline Logistic Regression pipeline
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

# Train ONLY on the training period
baseline_model.fit(X_train, y_train)

print("Baseline Model 1 trained successfully.")

Baseline Model 1 trained successfully.


In [ ]:
# Predictions on unseen test data
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("===== MODEL 1 BASELINE RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, y_pred))

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_test, y_pred))

===== MODEL 1 BASELINE RESULTS =====
Accuracy : 0.7907
Precision: 0.8472
Recall   : 0.8971
F1 Score : 0.8714
ROC-AUC  : 0.8783

===== CONFUSION MATRIX =====
[[ 7 11]
 [ 7 61]]

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.50      0.39      0.44        18
           1       0.85      0.90      0.87        68

    accuracy                           0.79        86
   macro avg       0.67      0.64      0.65        86
weighted avg       0.77      0.79      0.78        86



In [ ]:
# Extract the trained Logistic Regression model
classifier = baseline_model.named_steps["classifier"]

# Get coefficients
coefficients = classifier.coef_[0]

# Create a readable table
feature_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": coefficients
})

# Add absolute value to see strength
feature_importance["Absolute_Impact"] = (
    feature_importance["Coefficient"].abs()
)

# Sort by strongest impact
feature_importance = feature_importance.sort_values(
    "Absolute_Impact",
    ascending=False
)

display(feature_importance)

,Feature,Coefficient,Absolute_Impact
0,buyer_dbt_mean,1.488316,1.488316
1,buyer_dbt_sd,-0.130721,0.130721
3,buyer_promise_keep,0.124187,0.124187
5,terms,-0.049076,0.049076
6,quarter_end,0.010385,0.010385
4,amount_rel,0.001859,0.001859
2,buyer_dispute_rate,0.000000,0.000000


In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# Time-aware validation inside the training data
tscv = TimeSeriesSplit(n_splits=5)

C_values = [0.001, 0.01, 0.1, 1, 10, 100]

results = []

for C in C_values:

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=C,
            random_state=42,
            max_iter=1000
        ))
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=tscv,
        scoring="roc_auc"
    )

    results.append({
        "C": C,
        "Mean ROC-AUC": scores.mean(),
        "Std ROC-AUC": scores.std()
    })

results_df = pd.DataFrame(results)

display(results_df)

,C,Mean ROC-AUC,Std ROC-AUC
0,0.001,0.865733,0.070063
1,0.010,0.865378,0.064785
2,0.100,0.870432,0.061528
3,1.000,0.862554,0.068680
4,10.000,0.856563,0.068269
5,100.000,0.854389,0.066232


In [ ]:
tuned_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=0.1,
        random_state=42,
        max_iter=1000
    ))
])

# Train on the complete training period
tuned_model.fit(X_train, y_train)

print("Tuned Model 1 trained successfully.")

Tuned Model 1 trained successfully.


In [ ]:
# Predictions from tuned model on the untouched test set
tuned_pred = tuned_model.predict(X_test)
tuned_prob = tuned_model.predict_proba(X_test)[:, 1]

# Calculate metrics
tuned_accuracy = accuracy_score(y_test, tuned_pred)
tuned_precision = precision_score(y_test, tuned_pred)
tuned_recall = recall_score(y_test, tuned_pred)
tuned_f1 = f1_score(y_test, tuned_pred)
tuned_auc = roc_auc_score(y_test, tuned_prob)

print("===== MODEL 1 TUNED RESULTS =====")
print(f"Accuracy : {tuned_accuracy:.4f}")
print(f"Precision: {tuned_precision:.4f}")
print(f"Recall   : {tuned_recall:.4f}")
print(f"F1 Score : {tuned_f1:.4f}")
print(f"ROC-AUC  : {tuned_auc:.4f}")

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, tuned_pred))

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_test, tuned_pred))

===== MODEL 1 TUNED RESULTS =====
Accuracy : 0.8488
Precision: 0.8571
Recall   : 0.9706
F1 Score : 0.9103
ROC-AUC  : 0.8832

===== CONFUSION MATRIX =====
[[ 7 11]
 [ 2 66]]

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0       0.78      0.39      0.52        18
           1       0.86      0.97      0.91        68

    accuracy                           0.85        86
   macro avg       0.82      0.68      0.71        86
weighted avg       0.84      0.85      0.83        86



In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Get training probabilities from the tuned model
train_prob = tuned_model.predict_proba(X_train)[:, 1]

threshold_results = []

# Try thresholds from 0.30 to 0.80
for threshold in np.arange(0.30, 0.81, 0.01):

    train_pred_threshold = (
        train_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_train, train_pred_threshold),
        "Precision": precision_score(
            y_train,
            train_pred_threshold,
            zero_division=0
        ),
        "Recall": recall_score(
            y_train,
            train_pred_threshold,
            zero_division=0
        ),
        "F1": f1_score(
            y_train,
            train_pred_threshold,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

# Sort by F1 first, then accuracy
best_thresholds = threshold_df.sort_values(
    ["F1", "Accuracy"],
    ascending=False
).head(10)

display(best_thresholds)

,Threshold,Accuracy,Precision,Recall,F1
29,0.59,0.850299,0.888060,0.922481,0.904943
30,0.60,0.850299,0.888060,0.922481,0.904943
24,0.54,0.844311,0.860140,0.953488,0.904412
25,0.55,0.838323,0.859155,0.945736,0.900369
27,0.57,0.838323,0.869565,0.930233,0.898876
28,0.58,0.838323,0.875000,0.922481,0.898113
26,0.56,0.832335,0.858156,0.937984,0.896296
31,0.61,0.832335,0.885496,0.899225,0.892308
23,0.53,0.820359,0.836735,0.953488,0.891304
35,0.65,0.832335,0.904000,0.875969,0.889764


In [ ]:
# Apply the candidate threshold to the untouched test probabilities
threshold = 0.59

test_pred_059 = (tuned_prob >= threshold).astype(int)

threshold_accuracy = accuracy_score(y_test, test_pred_059)
threshold_precision = precision_score(y_test, test_pred_059)
threshold_recall = recall_score(y_test, test_pred_059)
threshold_f1 = f1_score(y_test, test_pred_059)

print("===== MODEL 1 | C=0.1 | THRESHOLD=0.59 =====")
print(f"Accuracy : {threshold_accuracy:.4f}")
print(f"Precision: {threshold_precision:.4f}")
print(f"Recall   : {threshold_recall:.4f}")
print(f"F1 Score : {threshold_f1:.4f}")

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, test_pred_059))

===== MODEL 1 | C=0.1 | THRESHOLD=0.59 =====
Accuracy : 0.8488
Precision: 0.9231
Recall   : 0.8824
F1 Score : 0.9023

===== CONFUSION MATRIX =====
[[13  5]
 [ 8 60]]


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rf_model = Pipeline([
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        class_weight=None,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest Model 1 trained successfully.")

Random Forest Model 1 trained successfully.


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

tscv = TimeSeriesSplit(n_splits=5)

rf_params = [
    {"n_estimators": 200, "max_depth": 3, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 3, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 5, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 7, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 5, "min_samples_leaf": 4},
    {"n_estimators": 500, "max_depth": 5, "min_samples_leaf": 2},
]

rf_results = []

for params in rf_params:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = RandomForestClassifier(
            random_state=42,
            n_jobs=-1,
            **params
        )

        model.fit(X_tr, y_tr)

        val_prob = model.predict_proba(X_val)[:, 1]

        if len(np.unique(y_val)) == 2:
            fold_scores.append(
                roc_auc_score(y_val, val_prob)
            )

    rf_results.append({
        **params,
        "Mean ROC-AUC": np.mean(fold_scores),
        "Std ROC-AUC": np.std(fold_scores)
    })

rf_results_df = pd.DataFrame(rf_results)

display(
    rf_results_df.sort_values(
        "Mean ROC-AUC",
        ascending=False
    )
)

,n_estimators,max_depth,min_samples_leaf,Mean ROC-AUC,Std ROC-AUC
4,300,NaN,2,0.862945,0.099611
3,300,7.0,2,0.858139,0.097224
2,300,5.0,2,0.856892,0.093505
6,500,5.0,2,0.852250,0.099062
0,200,3.0,2,0.851016,0.090614
1,300,3.0,2,0.847588,0.099753
5,300,5.0,4,0.839215,0.092647


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

tscv = TimeSeriesSplit(n_splits=5)

gb_params = [
    {"n_estimators": 50, "learning_rate": 0.03, "max_depth": 2},
    {"n_estimators": 100, "learning_rate": 0.03, "max_depth": 2},
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2},
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 150, "learning_rate": 0.03, "max_depth": 2},
    {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 200, "learning_rate": 0.03, "max_depth": 2},
]

gb_results = []

for params in gb_params:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = GradientBoostingClassifier(
            random_state=42,
            **params
        )

        model.fit(X_tr, y_tr)

        val_prob = model.predict_proba(X_val)[:, 1]

        if len(np.unique(y_val)) == 2:
            fold_scores.append(
                roc_auc_score(y_val, val_prob)
            )

    gb_results.append({
        **params,
        "Mean ROC-AUC": np.mean(fold_scores),
        "Std ROC-AUC": np.std(fold_scores)
    })

gb_results_df = pd.DataFrame(gb_results)

display(
    gb_results_df.sort_values(
        "Mean ROC-AUC",
        ascending=False
    )
)

,n_estimators,learning_rate,max_depth,Mean ROC-AUC,Std ROC-AUC
0,50,0.03,2,0.786213,0.130515
4,150,0.03,2,0.774519,0.147024
1,100,0.03,2,0.769171,0.136443
6,200,0.03,2,0.765240,0.150272
2,100,0.05,2,0.764902,0.153477
3,100,0.05,3,0.761831,0.175604
5,150,0.05,3,0.760695,0.164481


In [ ]:
import joblib
import os

# Create directory
os.makedirs("models", exist_ok=True)

# Save the trained Logistic Regression pipeline/model
joblib.dump(rf_model if False else model, "models/model1_logistic_regression.joblib")

print("Model 1 saved successfully!")

Model 1 saved successfully!
